In [1]:
import pandas as pd
import numpy as np
import pickle

import easyMPRA_20231120 as em
from _helper_functions import read_tsv
from _helper_functions import revcomp

loading libraries...
done!


# load data to process

In [4]:
# load file names
file2name={}

fn='ref/fastq-samplenames-1234567.tsv'
indir=''

for name,fastq in read_tsv(fn,pc=False,header=False):
    file2name[fastq]=name

for f,n in file2name.items(): print(f,n)

GE_CLUSTERS_66BP_DNA_R5_S4_L005_R1_001.fastq D5
GE_CLUSTERS_66BP_DNA_R6_S5_L005_R1_001.fastq D6
GE_CLUSTERS_66BP_DNA_R7_S6_L005_R1_001.fastq D7
GE_CLUSTERS_66BP_RNA_R5_S1_L005_R1_001.fastq R5
GE_CLUSTERS_66BP_RNA_R6_S2_L005_R1_001.fastq R6
GE_CLUSTERS_66BP_RNA_R7_S3_L005_R1_001.fastq R7
GE_CLUSTERS_66BP_DNA_R1_S5_L006_R1_001.fastq D1
GE_CLUSTERS_66BP_DNA_R2_S6_L006_R1_001.fastq D2
GE_CLUSTERS_66BP_DNA_R3_S7_L006_R1_001.fastq D3
GE_CLUSTERS_66BP_DNA_R4_S8_L006_R1_001.fastq D4
GE_CLUSTERS_66BP_RNA_R1_S1_L006_R1_001.fastq R1
GE_CLUSTERS_66BP_RNA_R2_S2_L006_R1_001.fastq R2
GE_CLUSTERS_66BP_RNA_R3_S3_L006_R1_001.fastq R3
GE_CLUSTERS_66BP_RNA_R4_S4_L006_R1_001.fastq R4


# write batch jobs 

In [5]:
######################
# batch script templates
######################

pythonScriptTemplate=\
f'''

import easyMPRA_20231003 as em

Sample2Bc2ReadCount={{}}


indir='{indir}'

fastq=indir + "{{fastq}}"

_=em.exp_1_count_barcodes(in_fastq=fastq,
                          out_path="{{out_path}}",
                          bcLength=25,
                          out_basename="{{out_basename}}",
                          progressReport=1e8,
                          fastqTotalLines=None,)'''

print(pythonScriptTemplate)



import easyMPRA_20231003 as em

Sample2Bc2ReadCount={}


indir=''

fastq=indir + "{fastq}"

_=em.exp_1_count_barcodes(in_fastq=fastq,
                          out_path="{out_path}",
                          bcLength=25,
                          out_basename="{out_basename}",
                          progressReport=1e8,
                          fastqTotalLines=None,)


In [6]:
qsubScriptTemplate=\
'''#!/bin/bash

### The queue (-q) to which the job is being submitted to:
### *condo is the queue for when we need more resources than we own,
### *glean is the free queue based on open resources on TSCC,
### *home  is our home queue that only farley lab members can use
### The account (-A) will always be farley-group
#PBS -q condo
#PBS -A farley-group

### Specify the name of the notebook job, output file, and error/log file;
### $PBS_JOBID embeds the job id in the script (helpful for debugging, but can be removed)
#PBS -N analyze-barcodes
#PBS -o {pyscript}.log
#PBS -e {pyscript}.err

### Request resources for the job, and the walltime for the job to run
### (note: ppn - processors per node; maximum ppn is 16)
### Generally try to run jobs with a single node and increase ppn
#PBS -l nodes=1:ppn=32
#PBS -l walltime=8:00:00

##### Exports all user environment variables to the job
#PBS -V

### Specify the email (can be multiple, separated by commas) to recieve
### emails/diagnostic information about the job; -m sets the mail settings:
### (n: no mail sent, a: mail sent when job aborted, b: mail sent when job begins,
### e: mail sent when job terminates)
#PBS -M solvason@eng.ucsd.edu
#PBS -m abe

#######################################################
# !!!!! DIAGNOSTIC STATEMENTS BELOW THIS HEADER !!!!! #
#######################################################

# Print out diagnostic information to have within the output file,
# This information can be removed or kept as desired
echo "Diagnostic Information:"
echo $PBS_JOBID
echo $PBS_JOBNAME
echo $PBS_NODELIST
echo $PBS_O_WORKDIR
echo $PBS_USER
echo ""

echo "Connected Login Node:"
echo $PBS_O_HOST
echo ""

##############################################
# !!!!! JOB COMMANDS BELOW THIS HEADER !!!!! #
##############################################

date

source ~/.bash_profile
source ~/.condainit
conda activate notebook

python {pyscript}

date
'''

In [8]:
##############################################
# generate batch scripts for each sample
##############################################

out_path='01-Barcode-Processing/'
Sample2Bc2ReadCount={}


for fastq,sampleName in file2name.items():

    print(sampleName,fastq)
    
    # python script
    outname  =f'{out_path}/{sampleName}.py'
    outscript=pythonScriptTemplate.replace('{fastq}',fastq).replace('{sampleName}',sampleName).replace('{out_path}',out_path).replace('{out_basename}',sampleName)
    with open(outname,'w') as f: f.write(outscript)
    
    # qsub script 
    outqsub  =f'{out_path}/GO.{sampleName}'
    outqsubscript=qsubScriptTemplate.replace('{pyscript}',outname)
    with open(outqsub,'w') as f: f.write(outqsubscript)


    # break

D5 GE_CLUSTERS_66BP_DNA_R5_S4_L005_R1_001.fastq
D6 GE_CLUSTERS_66BP_DNA_R6_S5_L005_R1_001.fastq
D7 GE_CLUSTERS_66BP_DNA_R7_S6_L005_R1_001.fastq
R5 GE_CLUSTERS_66BP_RNA_R5_S1_L005_R1_001.fastq
R6 GE_CLUSTERS_66BP_RNA_R6_S2_L005_R1_001.fastq
R7 GE_CLUSTERS_66BP_RNA_R7_S3_L005_R1_001.fastq
D1 GE_CLUSTERS_66BP_DNA_R1_S5_L006_R1_001.fastq
D2 GE_CLUSTERS_66BP_DNA_R2_S6_L006_R1_001.fastq
D3 GE_CLUSTERS_66BP_DNA_R3_S7_L006_R1_001.fastq
D4 GE_CLUSTERS_66BP_DNA_R4_S8_L006_R1_001.fastq
R1 GE_CLUSTERS_66BP_RNA_R1_S1_L006_R1_001.fastq
R2 GE_CLUSTERS_66BP_RNA_R2_S2_L006_R1_001.fastq
R3 GE_CLUSTERS_66BP_RNA_R3_S3_L006_R1_001.fastq
R4 GE_CLUSTERS_66BP_RNA_R4_S4_L006_R1_001.fastq


## load bc2counts

In [10]:
datadir='01-Barcode-Processing/'
Sample2Bc2ReadCount={}

fn ='ref/fastq-samplenames-1234567.tsv'    
    
for name,fn in read_tsv(fn,pc=False,header=False):
    
    if 'test' in name: continue

    if name in Sample2Bc2ReadCount: 
        raise ValueError('Sample duplicate detected')

    else:
        fn=f'{datadir}/{name}.Bc2ReadCount.pickle'        
        try:
            print('Reading',name,'...')
            with open(fn,'rb') as f: Bc2ReadCount=pickle.load(f)
            Sample2Bc2ReadCount[name]=Bc2ReadCount

            # print('detected! ',name)
            # with open(fn,'rb') as f: pass

        except FileNotFoundError:
            print('\nFILE NOT FOUND',fn)
            print()
            
del Bc2ReadCount

Reading D1 ...
Reading D2 ...
Reading D3 ...
Reading D4 ...
Reading R1 ...
Reading R2 ...
Reading R3 ...
Reading R4 ...
Reading D5 ...
Reading D6 ...
Reading D7 ...
Reading R5 ...
Reading R6 ...
Reading R7 ...


In [24]:
# save compiled dictionary for faster loading
fn=f'01-Barcode-Processing//combined-Sample2Bc2ReadCount.pydict.pickle'

# if writing...
# with open(fn,'wb') as f: pickle.dump(Sample2Bc2ReadCount,f,protocol=pickle.HIGHEST_PROTOCOL)

# if reading...
with open(fn,'rb') as f: Sample2Bc2ReadCount=pickle.load(f)
    

# load dict

In [21]:

fn='../0-dict/01-Assemble-Dictionary/3klib-pe150-locOnly-3-filt_ubc2en_filt.minUReadsRequired=15.maxMmReadsAcceptable=15.bcTrimLength=25.tsv'
    
ubc2en=em.ubc2en_from_tsv(fn)

In [22]:

f'{len(ubc2en):,}'

'50,656'

# process read data

In [23]:
enOrderSheet='../0-dict/ref/order-sheet.txt'
_,En2EnId=em.enhancerOrderSheet_from_tsv(enOrderSheet)


In [25]:

bcdf=\
em.exp_2_create_barcode_table(Sample2Bc2ReadCount, 
                              enOrderSheet,
                              ubc2en)

bcdf.head()

Initializing barcode table with dictionary sequences...

	50,656 barcodes detected in the dictionary

Adding barcode readcounts from D1...

        Number Barcodes Analyzed       = 5,861,580
        Number Barcodes in Dictionary  = 48,249 (0.8%)
        Number Reads Analyzed          = 77,939,298
        Number Reads in Dict Barcodes  = 1,040,896 (1.3%)
        

Adding barcode readcounts from D2...

        Number Barcodes Analyzed       = 5,659,168
        Number Barcodes in Dictionary  = 48,978 (0.9%)
        Number Reads Analyzed          = 66,332,201
        Number Reads in Dict Barcodes  = 891,745 (1.3%)
        

Adding barcode readcounts from D3...

        Number Barcodes Analyzed       = 8,165,911
        Number Barcodes in Dictionary  = 50,354 (0.6%)
        Number Reads Analyzed          = 129,858,134
        Number Reads in Dict Barcodes  = 1,747,658 (1.3%)
        

Adding barcode readcounts from D4...

        Number Barcodes Analyzed       = 6,296,362
        Number Bar

,barcode-seq,enhancer-seq,enhancer-id,rc-D1,rpm-D1,rc-D2,rpm-D2,rc-D3,rpm-D3,rc-D4,rpm-D4,rc-R1,rpm-R1,rc-R2,rpm-R2,rc-R3,rpm-R3,rc-R4,rpm-R4,rc-D5,rpm-D5,rc-D6,rpm-D6,rc-D7,rpm-D7,rc-R5,rpm-R5,rc-R6,rpm-R6,rc-R7,rpm-R7
0,AGTGACCTTAATCAGCTATCTAGCA,CTTATCACGTTAATATCGCCTTTTTTCCACCCTTTCCTGCTTTTCG...,Chr1:2114490-2114556,18.0,0.234075,16.0,0.244497,37.0,0.288813,21.0,0.238894,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,0.269333,35.0,0.309451,48.0,0.363696,NaN,NaN,5.0,0.020759,NaN,NaN
1,TACCCATTTTTCTTACCCTCGTTTG,TTTATCTTTCAGCCAATCTGGATCTTTTTCATGCTCTGCAAGATCT...,Chr9:3738803-3738869,16.0,0.208067,15.0,0.229216,34.0,0.265396,31.0,0.352654,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.0,0.269333,38.0,0.335975,41.0,0.310657,NaN,NaN,NaN,NaN,NaN,NaN
2,GCATATAGTACTGTACCCTATAGCG,TTGGATTCGTAGCCACCTATCTGAAGAGTATCCCCTTGACGTTTCA...,Chr4:2991141-2991207,27.0,0.351112,20.0,0.305621,68.0,0.530792,34.0,0.386781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,0.471333,38.0,0.335975,93.0,0.704661,NaN,NaN,NaN,NaN,NaN,NaN
3,AGGGCGACGGGGGCCTCACCTGTTC,TCTATCACTATTTAATATTAAACGCAGGGATCAATGGAATTCACAT...,Chr3:6949831-6949897,35.0,0.455146,18.0,0.275059,51.0,0.398093,51.0,0.580172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.0,0.377066,42.0,0.371340,58.0,0.439465,NaN,NaN,NaN,NaN,1836.0,7.815256
4,TTATTAAACAGCCGCCTTGCGATTA,CATATCCATGTAAAGAAAAGACAAACAGCCGATATAGGTAAGCTGA...,Chr2:2461673-2461739,NaN,NaN,2.0,0.030562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.013467,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# if writing
bcdf.to_csv('01-Barcode-Processing//bcdf.tsv',sep='\t',index=None)
bcdf.to_pickle(f'01-Barcode-Processing/bcdf.pd.df.pickle')

# if reading
# bcdf=pd.read_pickle('01b-generated-tables/20231201-bcdf-u15_m15_3-1M.pd.df.pickle')
# bcdf.head()

In [27]:
# only keep rpm columns
print('removing the following columns from the dataframe!!')
for c in [i for i in bcdf.columns if 'rc-' in i]:
    print(c)
    bcdf=bcdf.drop(c,axis=1)

removing the following columns from the dataframe!!
rc-D1
rc-D2
rc-D3
rc-D4
rc-R1
rc-R2
rc-R3
rc-R4
rc-D5
rc-D6
rc-D7
rc-R5
rc-R6
rc-R7


In [28]:
endf=em.exp_3_bcdf_to_endf(bcdf,outbasename=f'01-Barcode-Processing//endf')

Colappsing on enhancers... This may take some time...

Writing out endf with basename 01-Barcode-Processing//endf...

Done!


In [30]:
# if writing
endf.to_pickle(f'01-Barcode-Processing/endf.pd.df.pickle')